In [ ]:
print("Setup: imports and load empirical landscapes")

import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
import style_config as sc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import logomaker
import mavenn

OUT_DIR = "."


def standardize_theta(theta):
    theta = np.asarray(theta, dtype=float)
    centered = theta - theta.mean(axis=1, keepdims=True)
    scale = centered.std() * np.sqrt(theta.shape[0])
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Cannot standardize additive effects with zero variance.")
    return centered / scale


def random_sequence_sd(theta):
    return np.sqrt(np.mean(theta**2, axis=1).sum())

# Load lac promoter additive model (L=75, C=4)
lac_model = mavenn.load_example_model('sortseq_mpa_additive')
lac_theta_dict = lac_model.get_theta(gauge='consensus')
lac_theta = standardize_theta(lac_theta_dict['theta_lc'])
lac_alphabet = list(lac_model.alphabet)
L_lac, C_lac = lac_theta.shape
lac_positions = np.arange(-L_lac, 0)
print(f"lac promoter: L={L_lac}, C={C_lac}, sd(F)={random_sequence_sd(lac_theta):.3f}")

# Load GB1 protein additive model (L=55, C=20)
gb1_model = mavenn.load_example_model('gb1_ge_additive')
gb1_theta_dict = gb1_model.get_theta(gauge='consensus')
gb1_theta = standardize_theta(gb1_theta_dict['theta_lc'])
gb1_alphabet = list(gb1_model.alphabet)
L_gb1, C_gb1 = gb1_theta.shape
gb1_positions = np.arange(2, 2 + L_gb1)
gb1_wt_seq = gb1_model.x_stats['consensus_seq']
gb1_wt_idx = np.array([gb1_alphabet.index(aa) for aa in gb1_wt_seq])
if len(gb1_wt_seq) != L_gb1:
    raise ValueError("GB1 wild-type sequence length does not match theta.")
gb1_theta_display = gb1_theta - gb1_theta[np.arange(L_gb1), gb1_wt_idx][:, None]
print(f"GB1 protein: L={L_gb1}, C={C_gb1}, sd(F)={random_sequence_sd(gb1_theta):.3f}")

In [ ]:
print("Figure 1: empirical landscapes")

fig, axes = plt.subplots(1, 2, figsize=(sc.TEXT_WIDTH, 2.2), width_ratios=[1,1])

# --- Panel (a): lac promoter sequence logo ---
ax = axes[0]
logo_df = pd.DataFrame(lac_theta, columns=lac_alphabet, index=lac_positions)
logomaker.Logo(logo_df, ax=ax, color_scheme='classic', 
               font_name='Arial Rounded MT Bold',
               fade_below=0.5, shade_below=0.5,
               show_spines=False,
               center_values=True)
ax.set_xlim(lac_positions[0] - 0.5, lac_positions[-1] + 0.5)
ax.set_xticks(range(-70, 0, 10))
ax.set_xlabel('Position')
ax.set_ylabel(r'Effect')
ax.set_title(r'(a)  $\it{lac}$ promoter ($L=75$, $C=4$)',
             fontsize=sc.PANEL_TITLE_SIZE, loc='left')

# --- Panel (b): GB1 protein heatmap ---
ax = axes[1]
gb1_abs = np.abs(gb1_theta_display).max()
im = ax.imshow(gb1_theta_display.T, aspect='auto', cmap='PiYG',
               vmin=-gb1_abs,
               vmax=gb1_abs,
               interpolation='nearest',
               origin='upper',
               extent=(gb1_positions[0] - 0.5, gb1_positions[-1] + 0.5,
                       C_gb1 - 0.5, -0.5))
ax.scatter(gb1_positions, gb1_wt_idx, s=3, c='black', marker='o',
           linewidths=0, zorder=3)

# Label amino acid alphabet on y-axis
ax.set_yticks(range(C_gb1))
ax.set_yticklabels(gb1_alphabet, fontsize=5, ha='center')
ax.tick_params(axis='y', length=0)
ax.set_xticks(np.arange(10, 60, 10))
ax.set_xlabel('Position')
ax.set_title(r'(b)  GB1 protein ($L=55$, $C=20$)',
             fontsize=sc.PANEL_TITLE_SIZE, loc='left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

# Colorbar
cbar = fig.colorbar(im, ax=ax, **sc.COLORBAR_KW)
cbar.set_label(r'Effect (relative to WT)')
cbar.ax.tick_params(labelsize=5)
cbar.outline.set_visible(False)
for spine in cbar.ax.spines.values():
    spine.set_visible(False)

fig.tight_layout(w_pad=2)
fig.savefig(f"{OUT_DIR}/fig1.pdf", bbox_inches='tight')
fig.savefig(f"{OUT_DIR}/fig1.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved fig1.pdf and fig1.png")